In [11]:
import torch
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

GPU tersedia: True
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [10]:
import pandas as pd
import numpy as np
import optuna

from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib
import os


In [12]:
def check_dataset_eda():
    print("Membaca dataset chat_dataset_100.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset_100.csv")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    df = df.drop_duplicates(subset=['teks_chat'])
    print("\n" + "="*40)
    print("INFORMASI UMUM DATASET")
    print("="*40)
    print(f"Total baris data mentah : {len(df)}")
    
    missing_data = df.isnull().sum().sum()
    print(f"Total data kosong (NaN) : {missing_data}")
    
    df_clean = df.dropna()
    print(f"Total data bersih       : {len(df_clean)}")
    
    print("\n" + "="*40)
    print("KOLOM DALAM DATASET")
    print("="*40)
    print(list(df.columns))
    
    print("\n" + "="*40)
    print(" JUMLAH DATA BERDASARKAN KELAS (INTENT)")
    print("="*40)
    intent_counts = df_clean['label_intent'].value_counts()
    for intent, count in intent_counts.items():
        print(f"- {intent}: {count} baris")
        
    print(f"\nTotal jenis kelas (intent): {len(intent_counts)}")
    print("="*40)
    
    return df_clean

# Jalankan fungsi EDA
df_eda = check_dataset_eda()

Membaca dataset chat_dataset_100.csv...
Dataset tidak ditemukan di: c:\Users\andyc\Documents\a_skripsi\training\prethesis\data\chat_dataset_100.csv


# SVM

### SVM Before Tuning

In [4]:
def train_intent_model():
    print("Membaca dataset v2_chat_dataset_100.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "v2_chat_dataset_100.csv")

    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None

    df = pd.read_csv(dataset_path)
    df = df.dropna()

    X = df['teks_chat']
    y = df['label_intent']
   
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),    # Mengambil kata tunggal dan pasangan 2 kata (frasa)
        min_df=2,              # Buang kata typo yang cuma muncul 1 kali di seluruh dataset
        max_df=0.9,            # Buang kata yang terlalu sering muncul (seperti "dan", "di")
        sublinear_tf=True      # Menekan dominasi kata yang di-spam berkali-kali dalam 1 chat
    )

    svm_model = SVC(
        kernel='linear',       # Linear masih oke, tapi kita tambah C
        C=2.0,                 # Coba naikkan C (bisa diubah-ubah antara 0.1, 1, 2, atau 10)
        class_weight='balanced', # Menyeimbangkan penalti jika ada kelas minoritas
        probability=True
    )

    print("Melatih model NLU (TF-IDF + SVM)...")

    model = make_pipeline(tfidf, svm_model)
    model.fit(X_train, y_train)

    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")

    return model

train_intent_model()

Membaca dataset v2_chat_dataset_100.csv...
Total data latih: 640 | Total data uji: 161
Melatih model NLU (TF-IDF + SVM)...


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       0.54      0.65      0.59        20
    bluffing       0.58      0.75      0.65        20
    claiming       0.77      0.48      0.59        21
   defending       0.64      0.70      0.67        20
  deflecting       0.62      0.50      0.56        20
     neutral       0.83      0.75      0.79        20
  persuading       0.50      0.50      0.50        20
     probing       0.82      0.90      0.86        20

    accuracy                           0.65       161
   macro avg       0.66      0.65      0.65       161
weighted avg       0.66      0.65      0.65       161

Model berhasil disimpan di models/intent_classifier.pkl



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](8,)","['accusing','bluffing','claiming',...,'neutral','persuading','probing']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.9
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


### SVM after tuning

In [6]:
def train_intent_model():
    print("Membaca dataset v2_chat_dataset_100.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "v2_chat_dataset_100.csv")

    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
    
    df = pd.read_csv(dataset_path)
    df = df.dropna()

    X = df['teks_chat']
    y = df['label_intent']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    pipeline = make_pipeline(TfidfVectorizer(), SVC(probability=True, class_weight='balanced'))

    param_grid = {
        'tfidfvectorizer__analyzer': ['word', 'char_wb'], 
        'tfidfvectorizer__ngram_range': [(1, 2), (2, 4), (1, 3)], 
        'tfidfvectorizer__use_idf': [True, False],
        
        'svc__C': [0.1, 1, 2, 5, 10],
        'svc__kernel': ['linear', 'rbf'],   
        'svc__gamma': ['scale', 'auto'],
        'svc__class_weight': ['balanced', None]
    }
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_weighted', verbose=2, n_jobs=-1)

    grid_search.fit(X_train, y_train)

    print(f"\nKombinasi Terbaik Ditemukan: {grid_search.best_params_}")

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    print(classification_report(y_test, y_pred))

    os.makedirs("models", exist_ok=True)
    joblib.dump(best_model, "models/intent_classifier.pkl")

    return best_model

train_intent_model()

Membaca dataset v2_chat_dataset_100.csv...
Total data latih: 640 | Total data uji: 161
Fitting 3 folds for each of 480 candidates, totalling 1440 fits


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Kombinasi Terbaik Ditemukan: {'svc__C': 5, 'svc__class_weight': 'balanced', 'svc__gamma': 'scale', 'svc__kernel': 'rbf', 'tfidfvectorizer__analyzer': 'char_wb', 'tfidfvectorizer__ngram_range': (1, 3), 'tfidfvectorizer__use_idf': True}

--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       0.63      0.85      0.72        20
    bluffing       0.76      0.80      0.78        20
    claiming       0.84      0.76      0.80        21
   defending       0.86      0.90      0.88        20
  deflecting       0.72      0.65      0.68        20
     neutral       1.00      0.90      0.95        20
  persuading       0.94      0.80      0.86        20
     probing       0.85      0.85      0.85        20

    accuracy                           0.81       161
   macro avg       0.83      0.81      0.82       161
weighted avg       0.83      0.81      0.82       161



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](8,)","['accusing','bluffing','claiming',...,'neutral','persuading','probing']"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'char_wb'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


# NAIVE BAIYES

### NAIVE_BAYES Before Tuning

In [ ]:
def train_intent_model_nb():
    print("Membaca dataset chat_dataset_100.csv...")
    
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset_100.csv")    
    model_dir = os.path.join(base_dir, "models")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    # 1. Perbaikan pada train_test_split (tambah stratify=y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + Naive Bayes)...")
    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Uji Model Naive Bayes ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    # 2. Simpan model dengan path yang dinamis mengikuti model_dir
    os.makedirs(model_dir, exist_ok=True)
    model_save_path = os.path.join(model_dir, "intent_classifier_nb.pkl")
    
    joblib.dump(model, model_save_path)
    print(f"Model berhasil disimpan di {model_save_path}\n")
    
    return model

train_intent_model_nb()

In [ ]:
pipeline = make_pipeline(TfidfVectorizer(), MultinomialNB())

# 2. Tentukan Parameter yang mau diuji coba
# Ingat format penamaannya: nama_modul_huruf_kecil__nama_parameter
param_grid = {
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)], # Coba per-kata (unigram) dan gabungan 2 kata (bigram)
    'tfidfvectorizer__use_idf': [True, False],        # Coba aktifkan/nonaktifkan pembobotan IDF
    'multinomialnb__alpha': [0.1, 0.5, 1.0, 2.0]      # Parameter smoothing Naive Bayes. 1.0 adalah default.
}

# 3. Setup GridSearchCV
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=3,                  # Cross-validation: data dibagi 3 lipatan untuk dites
    scoring='f1_weighted', # Pakai f1_weighted karena proporsi intent mungkin tidak seimbang
    verbose=1,             # Menampilkan progress bar saat training
    n_jobs=-1              # Gunakan seluruh core CPU laptop untuk mempercepat proses
)

# 4. Eksekusi Pencarian
print("Memulai Grid Search (mencari setingan terbaik)...")
grid_search.fit(X_train, y_train)

# 5. Tampilkan Hasil dan Ambil Model Terbaik
print(f"\nKombinasi Parameter Terbaik: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

# Lanjut evaluasi pakai best_model
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

naive baiyes pakai optuna

In [ ]:
# 1. Definisikan Objective Function untuk Optuna
def objective(trial):
    # Tentukan ruang pencarian parameter yang mau diuji coba
    # Gunakan categorical untuk pilihan spesifik, dan float/int untuk rentang angka
    ngram_range = trial.suggest_categorical('ngram_range', [(1, 1), (1, 2)])
    use_idf = trial.suggest_categorical('use_idf', [True, False])
    
    # Kelebihan Optuna: Kita bisa kasih rentang angka (misal 0.1 sampai 2.0) 
    # tanpa harus mengetik satu-satu seperti di GridSearch
    alpha = trial.suggest_float('alpha', 0.1, 2.0)

    # Definisikan Pipeline di dalam fungsi dengan parameter dari trial
    pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=ngram_range, use_idf=use_idf), 
        MultinomialNB(alpha=alpha)
    )

    # Evaluasi menggunakan Cross Validation
    # cv=3, scoring='f1_weighted', n_jobs=-1 sama seperti sebelumnya
    score = cross_val_score(
        pipeline, 
        X_train, 
        y_train, 
        cv=3, 
        scoring='f1_weighted', 
        n_jobs=-1
    )
    
    # Kembalikan rata-rata skor dari 3 fold tersebut
    return score.mean()

# 2. Eksekusi Pencarian dengan Optuna
print("Memulai Bayesian Optimization (mencari setingan terbaik)...")
# direction='maximize' karena kita ingin skor f1_weighted sebesar mungkin
study = optuna.create_study(direction='maximize')

# n_trials adalah jumlah eksperimen. Semakin besar semakin bagus tapi lebih lama.
study.optimize(objective, n_trials=30) 

# 3. Tampilkan Hasil
print(f"\nSkor Terbaik: {study.best_value}")
print(f"Kombinasi Parameter Terbaik: {study.best_params}")

# 4. Ambil Model Terbaik dan Evaluasi Ulang
best_params = study.best_params
best_model = make_pipeline(
    TfidfVectorizer(ngram_range=best_params['ngram_range'], use_idf=best_params['use_idf']),
    MultinomialNB(alpha=best_params['alpha'])
)

# Lanjut evaluasi pakai best_model
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

# TRANSFORMER

In [ ]:
def train_intent_model_transformer():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
        
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None

    df = pd.read_csv(dataset_path).dropna()
    
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['label_intent'])
    num_labels = len(label_encoder.classes_)
    
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(df_train)} | Total data uji: {len(df_test)}")
    
    train_dataset = Dataset.from_pandas(df_train[['teks_chat', 'label']])
    test_dataset = Dataset.from_pandas(df_test[['teks_chat', 'label']])
    
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["teks_chat"], padding="max_length", truncation=True, max_length=128)
    
    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    
    print("Mengunduh/Memuat model IndoBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    training_args = TrainingArguments(
        output_dir="./models/transformer_results",
        eval_strategy="epoch",  
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,     
        weight_decay=0.01,
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {"accuracy": (predictions == labels).mean()}
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    
    print("Mulai melatih model Transformer...")
    trainer.train()
    
    print("\n--- Hasil Uji Model Transformer ---")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]
    
    target_names = label_encoder.classes_
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    model_save_path = "models"
    tokenizer.save_pretrained(model_save_path)
    model.save_pretrained(model_save_path)
    
    joblib.dump(label_encoder, f"{model_save_path}/label_encoder.pkl")
    
    print(f"Model berhasil disimpan di folder: {model_save_path}\n")
    return model, tokenizer, label_encoder

train_intent_model_transformer()


In [ ]:
def predict_intent(chat_text):
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [ ]:
train_intent_model_transformer()
# train_intent_model_nb()
# train_intent_model()
    
# print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
# test_chats = [
#     "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
#     "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
#     "Woy si Budi dari tadi diem aja, fix dia ketuanya"
# ]

# for chat in test_chats:
#     intent, prob = predict_intent(chat)
#     print(f"Chat Player: '{chat}'")
#     print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")